In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/EricGuo5513/momask-codes.git /content/momask-codes

Cloning into '/content/momask-codes'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 277 (delta 132), reused 92 (delta 92), pack-reused 102 (from 1)
Receiving objects: 100% (277/277), 1.15 MiB | 18.36 MiB/s, done.
Resolving deltas: 100% (143/143), done.


In [ ]:
%cd /content/momask-codes

/content/momask-codes


In [ ]:
# Environment adaptation: The original MoMask repository specifies older dependencies, including PyTorch 1.12.0 and NumPy 1.21.5. Our Colab runtime uses Python 3.13.15 and
# PyTorch 2.11.0+cu128 with a Tesla T4. We retained Colab’s PyTorch installation and selected  additional dependencies needed for inference.
# Compatibility will be verified through imports, pretrained checkpoint loading, motion generation, and output export.

%pip install einops ftfy regex tqdm gdown scipy matplotlib
%pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_8laiwox
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_8laiwox
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369549 sha256=8a1499d347b4217818b21327c1f443e415532452a05a78350ec14a1c8d90f988
  Stored in directory: /tmp/pip-ephem-wheel-cache-r0kuiuzm/wheels/cb/a8/74/5f32d6cf0407457f0f62737b6da5c14eb86b9cac476fdf630d
Successfully built clip


In [ ]:
# NumPy compatibility fix: Importing the BVH converter failed because the visualization code referenced the unavailable internal module
# `numpy.core.umath_tests`. In `visualization/Animation.py` and `visualization/Quaternions.py`, we removed that import and replaced
# `ut.matrix_multiply` with the public `np.matmul` function. This preserves batched matrix multiplication used for skeletal transformations.
from pathlib import Path

root = Path("/content/momask-codes")

for filename in ["Animation.py", "Quaternions.py"]:
    path = root / "visualization" / filename
    source = path.read_text()

    # Remove the obsolete import, preserving surrounding indentation.
    source = "\n".join(
        line for line in source.split("\n")
        if line.strip() != "import numpy.core.umath_tests as ut"
    )

    # Use NumPy's supported matrix multiplication.
    source = source.replace("ut.matrix_multiply(", "np.matmul(")

    path.write_text(source)
    print("Patched:", filename)

Patched: Animation.py
Patched: Quaternions.py


In [ ]:
# Removed NumPy alias: The original code used `np.float`, which is unavailable in the installed NumPy version. We replaced exact occurrences
# with Python’s built-in `float`, preserving explicit types such as `np.float32` and `np.float64`.
from pathlib import Path
import re

root = Path("/content/momask-codes")

for path in root.rglob("*.py"):
    source = path.read_text()

    # Match np.float exactly; preserve np.float32 and np.float64.
    updated = re.sub(r"\bnp\.float\b", "float", source)

    if updated != source:
        path.write_text(updated)
        print("Patched:", path.relative_to(root))

Patched: common/quaternion.py
Patched: utils/motion_process.py
Patched: visualization/remove_fs.py


In [ ]:
import clip
import numpy as np
import torch

from models.mask_transformer.transformer import MaskTransformer, ResidualTransformer
from models.vq.model import RVQVAE
from visualization.joints2bvh import Joint2BVHConvertor

print("Imports successful")
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

Imports successful
NumPy: 2.1.3
PyTorch: 2.11.0+cu130


/content/momask-codes/visualization/BVH_mod.py:113: SyntaxWarning: invalid escape sequence '\s'
  jmatch = re.match("\s*JOINT\s+(\w+:?\w+)", line)
/content/momask-codes/visualization/BVH_mod.py:126: SyntaxWarning: invalid escape sequence '\s'
  fmatch = re.match("\s*Frames:\s+(\d+)", line)
/content/momask-codes/visualization/BVH_mod.py:137: SyntaxWarning: invalid escape sequence '\s'
  fmatch = re.match("\s*Frame Time:\s+([\d\.]+)", line)


In [ ]:
# Pretrained checkpoints: We use the authors’ pretrained weights for inference. These include the motion tokenizer/decoder, masked
# transformer, residual transformer, and motion-length estimator. No training is performed in this milestone.
%cd /content/momask-codes
!bash prepare/download_models.sh

/content/momask-codes
Downloading...
From (original): https://drive.google.com/uc?id=1vXS7SHJBgWPt59wupQ5UUzhFObrnGkQ0
From (redirected): https://drive.google.com/uc?id=1vXS7SHJBgWPt59wupQ5UUzhFObrnGkQ0&confirm=t&uuid=07855702-3d48-4ceb-83d4-1daa53989711
To: /content/momask-codes/checkpoints/t2m/humanml3d_models.zip
100% 188M/188M [00:07<00:00, 26.2MB/s]
Unzipping humanml3d_models.zip
Archive:  humanml3d_models.zip
   creating: length_estimator/
   creating: length_estimator/model/
  inflating: length_estimator/model/finest.tar  
   creating: rvq_nq6_dc512_nc512_noshare_qdp0.2/
   creating: rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/
   creating: rvq_nq6_dc512_nc512_noshare_qdp0.2/model/
  inflating: rvq_nq6_dc512_nc512_noshare_qdp0.2/opt.txt  
  inflating: rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/mean.npy  
  inflating: rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/std.npy  
  inflating: rvq_nq6_dc512_nc512_noshare_qdp0.2/model/net_best_fid.tar  
   creating: t2m_nlayer8_nhead6_ld384_ff1024_cdp

In [ ]:
# Checkpoint
from pathlib import Path

root = Path("/content/momask-codes/checkpoints/t2m")

required = [
    "t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns/opt.txt",
    "t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns/model/latest.tar",
    "tres_nlayer8_ld384_ff1024_rvq6ns_cdp0.2_sw/opt.txt",
    "tres_nlayer8_ld384_ff1024_rvq6ns_cdp0.2_sw/model/net_best_fid.tar",
    "rvq_nq6_dc512_nc512_noshare_qdp0.2/opt.txt",
    "rvq_nq6_dc512_nc512_noshare_qdp0.2/model/net_best_fid.tar",
    "rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/mean.npy",
    "rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/std.npy",
    "length_estimator/model/finest.tar",
]

missing = []

for filename in required:
    path = root / filename
    exists = path.is_file() and path.stat().st_size > 0
    print("OK" if exists else "MISSING", filename)
    if not exists:
        missing.append(filename)

assert not missing, "Some checkpoint files are missing."
print("\nCheckpoint files are present.")

OK t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns/opt.txt
OK t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns/model/latest.tar
OK tres_nlayer8_ld384_ff1024_rvq6ns_cdp0.2_sw/opt.txt
OK tres_nlayer8_ld384_ff1024_rvq6ns_cdp0.2_sw/model/net_best_fid.tar
OK rvq_nq6_dc512_nc512_noshare_qdp0.2/opt.txt
OK rvq_nq6_dc512_nc512_noshare_qdp0.2/model/net_best_fid.tar
OK rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/mean.npy
OK rvq_nq6_dc512_nc512_noshare_qdp0.2/meta/std.npy
OK length_estimator/model/finest.tar

Checkpoint files are present.


In [ ]:
# Matplotlib compatibility: Motion generation completed, but MP4 rendering failed because the original code assigned empty lists to
# read-only axes properties. We replaced these assignments with removal of individual drawing objects and explicitly attached the 3D axes
# to the figure. The changes affect visualization, not the model architecture or weights.
from pathlib import Path
import re

path = Path("/content/momask-codes/utils/plot_script.py")
source = path.read_text()

# Replace assignments with removal of individual drawing objects.
for collection in ["lines", "collections"]:
    pattern = rf"^([ \t]*)ax\.{collection} = \[\][ \t]*$"

    def replacement(match, collection=collection):
        indent = match.group(1)
        return (
            f"{indent}for artist in list(ax.{collection}):\n"
            f"{indent}    artist.remove()"
        )

    source = re.sub(pattern, replacement, source, flags=re.MULTILINE)

# Explicitly attach the 3D axes to the figure.
source = source.replace(
    "ax = p3.Axes3D(fig)",
    'ax = fig.add_subplot(111, projection="3d")'
)

path.write_text(source)
print("Patched plotting code.")

Patched plotting code.


In [ ]:
# Run the model
!python gen_t2m.py \
    --gpu_id 0 \
    --dataset_name t2m \
    --ext milestone2_walk \
    --text_prompt "A person walks forward." \
    --motion_length 100 \
    --seed 42 \
    --time_steps 10 \
    --repeat_times 1

------------ Options -------------
batch_size: 32
checkpoints_dir: ./checkpoints
cond_scale: 4
dataset_name: t2m
dropout: 0.2
ext: milestone2_walk
ff_size: 1024
force_mask: False
gpu_id: 0
gumbel_sample: False
is_train: False
latent_dim: 384
mask_edit_section: None
max_motion_length: 196
motion_length: 100
n_heads: 6
n_layers: 8
name: t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns
num_batch: 2
repeat_times: 1
res_name: tres_nlayer8_ld384_ff1024_rvq6ns_cdp0.2_sw
seed: 42
source_motion: example_data/000612.npy
temperature: 1.0
text_path: 
text_prompt: A person walks forward.
time_steps: 10
topkr: 0.9
unit_length: 4
use_res_model: False
vq_name: rvq_nq1_dc512_nc512
which_epoch: latest
-------------- End ----------------
Reading ./checkpoints/t2m/t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns/opt.txt
Reading ./checkpoints/t2m/rvq_nq6_dc512_nc512_noshare_qdp0.2/opt.txt
Loading VQ Model rvq_nq6_dc512_nc512_noshare_qdp0.2 Completed!
Reading ./checkpoints/t2m/tres_nlayer8_ld384_ff1024_rvq6ns_cdp

In [ ]:
# Generated a 100-frame walking motion with seed 42 and 10 masked-generation iterations.
from pathlib import Path
from IPython.display import Video, display

root = Path("/content/momask-codes/generation/milestone2_walk")
videos = sorted(
    p for p in root.rglob("*.mp4")
    if not p.stem.endswith("_ik") and p.stat().st_size > 0
)

assert videos, "No non-empty video found—check the generation output."
print(videos[0])
display(Video(str(videos[0]), embed=True))

/content/momask-codes/generation/milestone2_walk/animations/0/sample0_repeat0_len100.mp4


In [ ]:
# Run the model
!python gen_t2m.py \
    --gpu_id 0 \
    --dataset_name t2m \
    --ext milestone2_wave \
    --text_prompt "A person waving his hand." \
    --motion_length 100 \
    --seed 42 \
    --time_steps 10 \
    --repeat_times 1

------------ Options -------------
batch_size: 32
checkpoints_dir: ./checkpoints
cond_scale: 4
dataset_name: t2m
dropout: 0.2
ext: milestone2_wave
ff_size: 1024
force_mask: False
gpu_id: 0
gumbel_sample: False
is_train: False
latent_dim: 384
mask_edit_section: None
max_motion_length: 196
motion_length: 100
n_heads: 6
n_layers: 8
name: t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns
num_batch: 2
repeat_times: 1
res_name: tres_nlayer8_ld384_ff1024_rvq6ns_cdp0.2_sw
seed: 42
source_motion: example_data/000612.npy
temperature: 1.0
text_path: 
text_prompt: A person waving his hand.
time_steps: 10
topkr: 0.9
unit_length: 4
use_res_model: False
vq_name: rvq_nq1_dc512_nc512
which_epoch: latest
-------------- End ----------------
Reading ./checkpoints/t2m/t2m_nlayer8_nhead6_ld384_ff1024_cdp0.1_rvq6ns/opt.txt
Reading ./checkpoints/t2m/rvq_nq6_dc512_nc512_noshare_qdp0.2/opt.txt
Loading VQ Model rvq_nq6_dc512_nc512_noshare_qdp0.2 Completed!
Reading ./checkpoints/t2m/tres_nlayer8_ld384_ff1024_rvq6ns_c

In [ ]:
# Generated a 100-frame waving hand motion with seed 42 and 10 masked-generation iterations.
from pathlib import Path
from IPython.display import Video, display

root = Path("/content/momask-codes/generation/milestone2_wave")
videos = sorted(
    p for p in root.rglob("*.mp4")
    if not p.stem.endswith("_ik") and p.stat().st_size > 0
)

assert videos, "No non-empty video found—check the generation output."
print(videos[0])
display(Video(str(videos[0]), embed=True))

/content/momask-codes/generation/milestone2_wave/animations/0/sample0_repeat0_len100.mp4


In [ ]:
!cd /content/momask-codes && git rev-parse HEAD
!python --version
!pip freeze > /content/requirements-colab.txt
import torch, numpy; print(torch.__version__, numpy.__version__)

94a6636c9c463b7a9414c3401a6f1b67e6c51824
Python 3.13.15
2.11.0+cu130 2.1.3


In [ ]:
!cd /content/momask-codes && zip -r /content/outputs.zip generation/ /content/requirements-colab.txt
from google.colab import files; files.download('/content/outputs.zip')

  adding: generation/ (stored 0%)
  adding: generation/milestone2_walk/ (stored 0%)
  adding: generation/milestone2_walk/animations/ (stored 0%)
  adding: generation/milestone2_walk/animations/0/ (stored 0%)
  adding: generation/milestone2_walk/animations/0/sample0_repeat0_len100_ik.mp4 (deflated 8%)
  adding: generation/milestone2_walk/animations/0/sample0_repeat0_len100.mp4 (deflated 8%)
  adding: generation/milestone2_walk/animations/0/sample0_repeat0_len100_ik.bvh (deflated 61%)
  adding: generation/milestone2_walk/animations/0/sample0_repeat0_len100.bvh (deflated 61%)
  adding: generation/milestone2_walk/joints/ (stored 0%)
  adding: generation/milestone2_walk/joints/0/ (stored 0%)
  adding: generation/milestone2_walk/joints/0/sample0_repeat0_len100.npy (deflated 6%)
  adding: generation/milestone2_walk/joints/0/sample0_repeat0_len100_ik.npy (deflated 6%)
  adding: generation/milestone2_wave/ (stored 0%)
  adding: generation/milestone2_wave/animations/ (stored 0%)
  adding: genera

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login
login()   # paste your Write token in the box (https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import create_repo, upload_folder
create_repo("pgarces/momask-checkpoints", repo_type="model", exist_ok=True)
upload_folder(folder_path="/content/momask-codes/checkpoints/t2m",
              repo_id="pgarces/momask-checkpoints", repo_type="model", path_in_repo="t2m")

CommitInfo(commit_url='https://huggingface.co/pgarces/momask-checkpoints/commit/36cbddb91492238ce500e8633d86229f54f376b4', commit_message='Upload folder using huggingface_hub', commit_description='', oid='36cbddb91492238ce500e8633d86229f54f376b4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pgarces/momask-checkpoints', endpoint='https://huggingface.co', repo_type='model', repo_id='pgarces/momask-checkpoints'), pr_revision=None, pr_num=None)